This notebook is structured as follows:
- Importing the data
- Pre-processing the data \
 --> (Feature processing 1) Selecting features to keep, according to missing data, expected key features and expected useless features \
 --> (Feature processing 2) Finding better representations of the features to feed the model \
 --> (à dvlper!) Standardization, imputation ... \
- Training the model
- Evaluating performance
- (à dvlper!) Analysing the results \
<span style="color:rgb(255, 0, 90)">--> !!! What I think we should add to have make our project 1 amazing</span>

In [1]:
import numpy as np
from helpers import *

# Importing the data

In [2]:
# You have to change the path for it to work
data_path = r'C:\Users\natha\Documents\EPFL\Cours_MA1\ML\ML_course\projects\project1 - withGit\data\dataset'

In [3]:
x_train, x_test, y_train, train_ids, test_ids, headers_train = load_csv_data(data_path, sub_sample=False)

test


In [4]:
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(len(headers_train))

(328135, 321)
(109379, 321)
(328135,)
322


Solve the issue of "headers_train" having one more column than x_train:

In [5]:
print(headers_train)

['Id', '_STATE', 'FMONTH', 'IDATE', 'IMONTH', 'IDAY', 'IYEAR', 'DISPCODE', 'SEQNO', '_PSU', 'CTELENUM', 'PVTRESD1', 'COLGHOUS', 'STATERES', 'CELLFON3', 'LADULT', 'NUMADULT', 'NUMMEN', 'NUMWOMEN', 'CTELNUM1', 'CELLFON2', 'CADULT', 'PVTRESD2', 'CCLGHOUS', 'CSTATE', 'LANDLINE', 'HHADULT', 'GENHLTH', 'PHYSHLTH', 'MENTHLTH', 'POORHLTH', 'HLTHPLN1', 'PERSDOC2', 'MEDCOST', 'CHECKUP1', 'BPHIGH4', 'BPMEDS', 'BLOODCHO', 'CHOLCHK', 'TOLDHI2', 'CVDSTRK3', 'ASTHMA3', 'ASTHNOW', 'CHCSCNCR', 'CHCOCNCR', 'CHCCOPD1', 'HAVARTH3', 'ADDEPEV2', 'CHCKIDNY', 'DIABETE3', 'DIABAGE2', 'SEX', 'MARITAL', 'EDUCA', 'RENTHOM1', 'NUMHHOL2', 'NUMPHON2', 'CPDEMO1', 'VETERAN3', 'EMPLOY1', 'CHILDREN', 'INCOME2', 'INTERNET', 'WEIGHT2', 'HEIGHT3', 'PREGNANT', 'QLACTLM2', 'USEEQUIP', 'BLIND', 'DECIDE', 'DIFFWALK', 'DIFFDRES', 'DIFFALON', 'SMOKE100', 'SMOKDAY2', 'STOPSMK2', 'LASTSMK2', 'USENOW3', 'ALCDAY5', 'AVEDRNK2', 'DRNK3GE5', 'MAXDRNKS', 'FRUITJU1', 'FRUIT1', 'FVBEANS', 'FVGREEN', 'FVORANG', 'VEGETAB1', 'EXERANY2', 'EXR

In [6]:
headers_train.remove('Id')
print(len(headers_train))

321


# Pre-processing the data

### Remove columns with too many missing values

In [7]:
def find_missing_values(data, headers=None):
    """
    Finds the count of missing (NaN) values in each column of the dataset.

    Parameters:
    data (np.ndarray): The input NumPy array containing the data.
    headers (list of str, optional): List of column names in the dataset. If provided, column names will be used in the output.
                                      If not provided, column indices will be used.

    Returns:
    dict: A dictionary where keys are column names (if headers provided) or column indices, and values are the count of missing
    values (NaNs) in the respective columns.
    """
    num_rows, num_cols = data.shape
    missing_count = np.zeros(num_cols, dtype=int)  
    columns = np.linspace(0,num_cols, num_cols+1)
    
    for col in range(num_cols):
        # Count the missing values
        missing_count[col] = np.sum(np.isnan(data[:, col]))
    if headers : 
        # Returning only the columns with missing values
        missing_info = {headers[col]: missing_count[col] for col in range(num_cols) if missing_count[col] > 0}
    else : 
        missing_info = {columns[col]: missing_count[col] for col in range(num_cols) if missing_count[col] > 0}
    return missing_info

We fix a threshold of 10% inside the function remove_high_missing_columns. We choose 10% because we assume that if there are more missing values than that, the feature becomes unreliable for prediction. \
<span style="color:rgb(255, 0, 90)">This threshold must not be modified, because my data modification function that considers each feature individually was not specifically designed to account for columns with a higher number of missing values that are not in columns_to_keep. </span>


In [8]:
def remove_high_missing_columns(data, headers=None, headers_to_keep=None, headers_to_remove=None):
    """
    Removes columns with high missing values from the dataset, with options to automatically keep or remove specific columns.

    Parameters:
    data (np.ndarray): The input NumPy array containing the data.
    headers (list of str): List of all column names in the dataset.
    headers_to_keep (list of str): List of column names to keep regardless of missing values.
    headers_to_remove (list of str): List of column names to remove regardless of missing values.

    Returns:
    np.ndarray: A filtered NumPy array with the remaining columns.
    list of str: The list of remaining headers.
    """
    assert data.shape[1] == len(headers)
    
    num_rows, num_cols = data.shape
    threshold = num_rows / 10

    # Find the missing values count for each column
    missing_count = find_missing_values(data, headers)
    
    # Determine the indices of columns to automatically keep and remove
    indices_to_keep = [headers.index(col) for col in headers_to_keep if col in headers] if headers_to_keep else []
    indices_to_remove = [headers.index(col) for col in headers_to_remove if col in headers] if headers_to_remove else []
    
    # Create the mask for columns that meet the criteria
    columns_to_keep = [
        col for col in range(num_cols)
        if (
            (headers[col] in headers_to_keep) or
            (headers[col] not in headers_to_remove and missing_count.get(headers[col], 0) <= threshold)
        )
    ]

    # Filter the data and headers to only include the columns that meet the criteria
    filtered_data = data[:, columns_to_keep]
    remaining_headers = [headers[col] for col in columns_to_keep]

    return filtered_data, remaining_headers

There are some features that should be quite meaningful when trying to predict _MICHD, and others that should absolutely not. We keep the meaningful ones unconditionally of their missing values count (maybe there could  be a strong weight given to any response that is non NaN) and we make sure we get rid of the meaningless ones (like day or year). It is true that some factors like geographical location could be related to _MICHD (for example if one region shows a greater average of _MICHD) but we do not want our model to consider that in its predictions.

In [48]:
columns_to_keep = ['AGE', 'SEX', '_RACE', '_HISPANC', 'GENHLTH', 'PHYSHLTH', 'MENTHLTH', 'POORHLTH', 'BPHIGH4', 'BLOODCHO', 'CHOLCHK', 'TOLDHI2',
                                   'CVDINFR4', 'CVDCRHD4', 'CVDSTRK3', 'ASTHMA3', 'DIABETE3', 'DIABAGE2', 'DIABEDU', 'BLDSUGAR', 'INSULIN', 'PDIABTST', 'PREDIAB1',
                                   'DOCTDIAB', 'SMOKE100', 'SMOKDAY2', '_SMOKER3', 'STOPSMK2', 'ALCDAY5', 'AVEDRNK2', 'DRNK3GE5', 'EXERANY2', '_FRUTSUM', 'METVL11',
                                   'FC60_', 'MINAC11', 'WEIGHT2', '_BMI5', 'HAVARTH3', 'BPMEDS', 'EXERHMM1']


columns_to_remove = ['Id', 'FMONTH', 'IDATE', 'IMONTH', 'IDAY', 'IYEAR', 'DISPCODE', 'SEQNO', '_STATE', '_PSU', '_STSTR', 'HHADULT', 'CPDEMO1', 'EMPLOY1', 'CHILDREN', 
    'INTERNET', 'SEATBELT', 'IMFVPLAC', 'QSTVER', 'MSCODE', '_LLCPWT', '_STRWT', '_RAWRAKE', '_WT2RAKE', 'CLLCPWT', 'DUALCOR', 'WTKG3']

In [50]:
sliced_x_train, sliced_features = remove_high_missing_columns(x_train, headers_train, columns_to_keep, columns_to_remove)

In [51]:
sliced_x_train.shape

(328135, 135)

In [12]:
print(sliced_features)
print(len(sliced_features))

['GENHLTH', 'PHYSHLTH', 'MENTHLTH', 'POORHLTH', 'HLTHPLN1', 'PERSDOC2', 'MEDCOST', 'CHECKUP1', 'BPHIGH4', 'BPMEDS', 'BLOODCHO', 'CHOLCHK', 'TOLDHI2', 'CVDSTRK3', 'ASTHMA3', 'CHCSCNCR', 'CHCOCNCR', 'CHCCOPD1', 'HAVARTH3', 'ADDEPEV2', 'CHCKIDNY', 'DIABETE3', 'DIABAGE2', 'SEX', 'MARITAL', 'EDUCA', 'RENTHOM1', 'VETERAN3', 'INCOME2', 'WEIGHT2', 'HEIGHT3', 'QLACTLM2', 'USEEQUIP', 'BLIND', 'DECIDE', 'DIFFWALK', 'DIFFDRES', 'DIFFALON', 'SMOKE100', 'SMOKDAY2', 'STOPSMK2', 'USENOW3', 'ALCDAY5', 'AVEDRNK2', 'DRNK3GE5', 'FRUITJU1', 'FRUIT1', 'FVBEANS', 'FVGREEN', 'FVORANG', 'VEGETAB1', 'EXERANY2', 'EXERHMM1', 'STRENGTH', 'FLUSHOT6', 'PNEUVAC3', 'HIVTST6', 'PDIABTST', 'PREDIAB1', 'INSULIN', 'BLDSUGAR', 'DOCTDIAB', 'DIABEDU', 'QSTLANG', '_DUALUSE', '_RFHLTH', '_HCVU651', '_RFHYPE5', '_CHOLCHK', '_LTASTH1', '_CASTHM1', '_ASTHMS1', '_DRDXAR1', '_PRACE1', '_MRACE1', '_HISPANC', '_RACE', '_RACEG21', '_RACEGR3', '_RACE_G1', '_AGEG5YR', '_AGE65YR', '_AGE80', '_AGE_G', 'HTIN4', 'HTM4', '_BMI5', '_BMI5CAT',

## Modify the data to feed our model

A crucial step, because it is well known that Garbage In = Garbage Out ;)

As of now, many features of the dataset have values that are hard for the model to interpret.
For example, some use the value "9" to indicate that the responder did not know the answer to a question. This value could badly influence the training, as 9 is a scalar, that will weight against the binary values 0 and 1. Therefore, we replace 9 by NaN and we will then see how to treat the NaN's.

We have read through all the documentation of the 300+ features, to see how to make their values clearer for a machine learning application. A set of rules was written below, and then implemented in the function "replace_specific_values".

In [13]:
#Some of those rules are:

#DIABETE3 replace 2 by 1, replace 3, 4, 7, 9 by 0
#SMOKDAY2 replace 2 by 1, replace 3, 7, 9 by 0
#USENOW3 replace 2 by 1, replace 3, 7, 9 by 0
#ALCDAY5 replace 777, 888, 999 by 0
#AVEDRNK replace 77, 99 by 0
#DRNK3GE5 replace 77, 88, 99 by 0
#PREDIAB1 replace 2 by 1, replace 3, 7, 9 by 0
#BLDSUGAR if starts by 1, two last*365. If starts by 2, two last*52. If starts by 3, two last*12, if starts by 4, two last. If > 499, 0.
#DOCTDIAB if >76, replace by 0
#FC60_ replace 99900 by average of rest
#Some sets have 1,2,3,4,5,6,7,9 should replace the 9
#MARITAL if not 1, replace by 0
#PHYSHLTH replace 88, 77, 99 by 0
#POORHLTH replace 88, 77, 99 by 0
#PERSDOC2 replace 2 by 1, replace 3, 7, 9 by 0
#SEX replace 2 by 0
#EDUCA replace 9 by 0
#INCOME2 replace 77 and 99 by 4
#WEIGHT2 if starts by 9, replace by 162
#LASTSMK2 replace 77, 99 by Nan
#MAXDRNKS replace 77, 99 by 0
#FRUITJU1 if >= 300 replace by 0, if < 300 replace by 1
#FRUIT1 if >= 300 replace by 0, if < 300 replace by 1
#FVBEANS if >= 300 replace by 0, if < 300 replace by 1
#FVORANG if >= 300 replace by 0, if < 300 replace by 1
#VEGETAB1 if >= 300 replace by 0, if < 300 replace by 1
#EXERHMM1 replace by 1, 2, 3, 4 if 1<30<100<200
#JOINPAIN replace 77 and 99 by 0
#_ASTHMS1 replace 2 by 1, replace 3, 9 by 0
#_PRACE1 MAKE THE SPLIT
#_MRACE1 MAKE THE SPLI
#_RACE MAKE THE SPLIT
#_RACEGR3 MAKE THE SPLIT
#_RACE_G1 MAKE THE SPLIT
#_AGEG5YR replace 14 by 4
#_AGE65YR replace 2, 3 by 0
#HTIN4 replace blank by 67
#_BMI5 replace Nan by average of rest
#_CHLDCNT replace 9 by 0
#_EDUCAG replace 9 by 1
#_SMOKER3 replace 9 by 4
#DROCDY3_ replace 900 by 0
#_DRNKWEK replace 99900 by 0
#_FRUITEX replace 2 by 1
#_VEGETEX replace 2 by 1
#MAXVO2_ replace 99900 by average of rest
#STRFREQ_ replace 99900 by average of rest
#_PACAT1 replace 9 by 4 (or average of rest)
#_PA150R2 replace 2, 3 and 9 by 0
#_PA300R2 replace 2, 3 and 9 by 0
#_PAREC1 replace 2 and 3 by 1, replace 4 and 9 by 0
#_PASTAE1 replace 2 and 9 by 0
#_LMTACT1 replace 9 by 3
#_LMTWRK1 replace 9 by 3
#_LMTWRK1 replace 9 by 4
#_RFSEAT2 replace 9 by 2
#_PNEUMO2 replace 9 by 0

A function to check if the column has specific values:

In [14]:
def unique_values(data, column_name, headers):
    """
    Returns a sorted list of unique values from a specified column, excluding NaN values.

    Parameters:
    data (np.ndarray): The input NumPy array containing the data.
    column_name (str): The name of the column to analyze.
    headers (list of str): List of all column names in the dataset.

    Returns:
    list: A sorted list of unique values from the specified column, excluding NaNs.
    """
    if column_name not in headers:
        raise ValueError(f"Column '{column_name}' not found in headers.")
    
    # Get the index of the specified column
    col_index = headers.index(column_name)
    
    # Extract the column values and filter out NaN values
    column = data[:, col_index]
    non_nan_values = column[~np.isnan(column)]
    
    # Get the unique values and sort them in ascending order
    unique_values = sorted(set(non_nan_values))
    
    return unique_values


Test it:

In [15]:
print(unique_values(sliced_x_train, 'BLOODCHO', sliced_features))

[1.0, 2.0, 7.0, 9.0]


The large function to replace the values according to the set of rules:

In [16]:
def replace_specific_values(old_data, headers):
    """
    Iterates through each column in the dataset and performs an action if the unique values of the column match the target values.

    Parameters:
    old_data (np.ndarray): The input NumPy array containing the data.
    headers (list of str): List of all column names in the dataset.

    Returns:
    data (np.ndarray): The output NumPy array with the modified values.
    """
    data = np.copy(old_data)

    for column_name in headers:
        # Get unique values from the column
        unique_vals = unique_values(data, column_name, headers)

        if unique_vals == [1, 2, 9]:
            # Find the column index
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column == 2, 0, column)
            column = np.where(column == 9, np.nan, column)
            
            # Update the data with modified column
            data[:, col_idx] = column
            

        elif unique_vals == [1, 2, 7, 9]:
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column == 2, 0, column)
            column = np.where(column == 7, np.nan, column)
            column = np.where(column == 9, np.nan, column)
            
            data[:, col_idx] = column
            

        elif unique_vals == [1, 2, 3, 9]:
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column == 2, 1, column)
            column = np.where(column == 3, 0, column)
            column = np.where(column == 9, np.nan, column)
            
            data[:, col_idx] = column
             
        
        elif unique_vals == [1, 2, 3, 7, 9]:
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column == 2, 0, column)
            column = np.where(column == 3, 0, column)
            column = np.where(column == 7, np.nan, column)
            column = np.where(column == 9, np.nan, column)
            
            data[:, col_idx] = column

        elif unique_vals == [1, 2, 3, 4, 9] and column_name != '_PACAT1':
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column == 2, 1, column)
            column = np.where(column == 3, 1, column)
            column = np.where(column == 4, 0, column)
            column = np.where(column == 9, np.nan, column)
            
            data[:, col_idx] = column

        elif unique_vals == [1, 2, 3, 4, 7, 9]:
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column == 2, 0, column)
            column = np.where(column == 3, 0, column)
            column = np.where(column == 4, 0, column)
            column = np.where(column == 7, np.nan, column)
            column = np.where(column == 9, np.nan, column)
            
            data[:, col_idx] = column

        elif unique_vals == [1, 2, 3, 4, 5, 6, 7, 9]:
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column == 9, np.nan, column)

            data[:, col_idx] = column

        elif unique_vals == [1, 2, 3]:
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column == 3, np.nan, column)
            
            data[:, col_idx] = column
        
        elif column_name in ['ALCDAY5', 'EXERHMM1', 'STRENGTH']:
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column == 888, 0, column)
            column = np.where(column == 777, np.nan, column)
            column = np.where(column == 999, np.nan, column)

            data[:, col_idx] = column

        elif column_name in ['AVEDRNK2', 'INCOME2', 'LASTSMK2', 'MAXDRNKS', 'JOINPAIN']:
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column == 77, np.nan, column)
            column = np.where(column == 99, np.nan, column)

            data[:, col_idx] = column

        elif column_name in ['DRNK3GE5', 'PHYSHLTH', 'MENTHLTH', 'POORHLTH', 'DOCTDIAB']:
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column == 88, 0, column)
            column = np.where(column == 77, np.nan, column)
            column = np.where(column == 99, np.nan, column)

            data[:, col_idx] = column

        elif column_name == '_PACAT1':
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column == 9, np.nan, column)

            data[:, col_idx] = column

        elif column_name in ['FC60_', '_DRNKWEK', 'MAXVO2_', 'STRFREQ_']:
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column == 88, 0, column)
            column = np.where(column == 77, np.nan, column)
            column = np.where(column == 99900, np.nan, column)
            column = np.where(column == 999, np.nan, column)

            data[:, col_idx] = column

        elif column_name in ['EDUCA', '_CHLDCNT', '_EDUCAG', 'SMOKER3', '_INCOMG', 'PAMISS1_']:
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column == 9, np.nan, column)

            data[:, col_idx] = column

        elif column_name in ['_FRUITEX', '_VEGETEX']:
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column == 2, np.nan, column)

            data[:, col_idx] = column

        elif column_name == '_AGEG5YR':
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column == 14, np.nan, column)

            data[:, col_idx] = column
        
        elif column_name == 'DROCDY3_':
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column == 900, np.nan, column)

            data[:, col_idx] = column

        elif column_name == 'MARITAL':
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column != 1, 0, column)

            data[:, col_idx] = column

        elif column_name in ['WEIGHT2', 'HEIGHT3']:
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column == 777, np.nan, column)
            column = np.where(column == 7777, np.nan, column)
            column = np.where((column >= 9000) & (column // 1000 == 9), np.nan, column)

            data[:, col_idx] = column

        elif column_name in ['FRUITJU1', 'FRUIT1', 'FVBEANS', 'FVGREEN', 'FVORANG', 'VEGETAB1']:
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column >= 300, 1, 0)

            data[:, col_idx] = column

        elif column_name == 'BLDSUGAR':
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column == 888, 0, column)
            column = np.where(column == 777, np.nan, column)
            column = np.where(column == 999, np.nan, column)

            column_str = column.astype(str)
            mask_1 = np.char.startswith(column_str, '1')
            column[mask_1] = np.char.zfill(column_str[mask_1], 3).astype(int) % 100
            mask_2 = np.char.startswith(column_str, '2')
            column[mask_2] = (np.char.zfill(column_str[mask_2], 3).astype(int) % 100) / 7
            mask_3 = np.char.startswith(column_str, '3')
            column[mask_3] = (np.char.zfill(column_str[mask_3], 3).astype(int) % 100) / 30.4
            mask_4 = np.char.startswith(column_str, '4')
            column[mask_4] = (np.char.zfill(column_str[mask_4], 3).astype(int) % 100) / 365.25

            data[:, col_idx] = column

        elif column_name == 'EXERHMM1':
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column == 888, 0, column)
            column = np.where(column == 777, np.nan, column)
            column = np.where(column == 999, np.nan, column)

            column_str = column.astype(str)
            greater_than_100_mask = column.astype(float) >= 100
            column_str_gt_100 = column_str[greater_than_100_mask]
    
            column[greater_than_100_mask] = (
            np.char.zfill(column_str_gt_100, 3).astype(int) // 100 * 60 +
            np.char.zfill(column_str_gt_100, 3).astype(int) % 100
            )

            data[:, col_idx] = column
        
    return data

In [17]:
modified_data = replace_specific_values(sliced_x_train, sliced_features)

To check that we didn't neglect any modification, we write a function that looks for the columns that haven't been accessed during the data modification.

In [18]:
def get_unmodified_headers(old_data, new_data, headers):
    """
    Compares the old dataset with the new dataset and returns the headers 
    that have not been modified.

    Parameters:
    old_data (np.ndarray): The original NumPy array containing the old data.
    new_data (np.ndarray): The modified NumPy array containing the new data.
    headers (list of str): List of all column names in the dataset.

    Returns:
    list: A list of headers that have not been modified.
    """
    # Check the types of old_data and new_data
    if not isinstance(old_data, np.ndarray):
        raise ValueError("old_data must be a numpy ndarray")
    if not isinstance(new_data, np.ndarray):
        raise ValueError("new_data must be a numpy ndarray")
    
    if old_data.shape[1] != new_data.shape[1]:
        raise ValueError("old_data and new_data must have the same number of columns")
    
    unmodified_headers = []

    for i, column_name in enumerate(headers):
        # Check if the column in the old data is the same as in the new data
        if np.array_equal(old_data[:, i], new_data[:, i]):
            unmodified_headers.append(column_name)

    return unmodified_headers

In [19]:
intact_columns = get_unmodified_headers(sliced_x_train, modified_data, sliced_features)
print(intact_columns)

['SEX', '_PRACE1', '_MRACE1', '_RACE', '_RACEGR3', '_AGE80', '_AGE_G', '_MISFRTN', '_MISVEGN', '_FRTRESP', '_VEGRESP', '_FRT16', '_VEG23']


The columns that should indeed not be accessed are :
['SEX', '_AGE80', '_AGE_G', 'MISFRTN', '_MISVEGN', '_FRTRESP', '_VEGRESP', '_FRT16', '_VEG23', ].

We can notice that there are some specific features that have still not been adressed: those relating to "Race". For them, the scalar vales have no real order, so what we should do instead is "One-hot encoding" in order to better reflect the categorical nature of those features:

In [20]:
def one_hot_encode(data, headers, headers_to_encode):
    """
    One-hot encodes specified columns in the dataset without modifying the original data.

    Parameters:
    data (np.ndarray): The input NumPy array containing the data.
    headers (list of str): List of all column names in the dataset.
    headers_to_encode (list of str): List of column names to one-hot encode.

    Returns:
    np.ndarray: The new dataset with one-hot encoded columns.
    list of str: The updated headers list with new column names.
    """
    # Create a copy of the data to avoid modifying the original dataset
    data_copy = np.copy(data)
    
    # List to store new columns and headers
    new_columns = []
    new_headers = []

    if data_copy.shape[1] != len(headers):
        raise ValueError("The number of headers does not match the number of columns in the data.")

    # Iterate through all columns
    for column_name in headers:
        if column_name in headers_to_encode:
            # Get the index of the column
            col_idx = headers.index(column_name)
            
            # Get the sorted unique values of the column
            unique_vals = unique_values(data_copy, column_name, headers)
            
            # Create a one-hot column for each unique value
            for val in unique_vals:
                one_hot_column = np.where(data_copy[:, col_idx] == val, 1, 0)
                new_columns.append(one_hot_column)
                new_headers.append(f"{column_name}_{int(val)}")
        else:
            # If not one-hot encoding, add the original column
            col_idx = headers.index(column_name)
            new_columns.append(data_copy[:, col_idx])
            new_headers.append(column_name)

    # Stack all the new columns horizontally
    data_modified = np.column_stack(new_columns)
    
    return data_modified, new_headers

In [21]:
headers_to_encode = ['_PRACE1', '_MRACE1', '_RACE', '_RACEGR3', '_RACE_G1']
encoded_data, encoded_headers = one_hot_encode(modified_data, sliced_features, headers_to_encode)

print("Original Data Shape:", sliced_x_train.shape)
print("Encoded Data Shape:", encoded_data.shape)
print("Updated Headers Length:", len(encoded_headers))


Original Data Shape: (328135, 135)
Encoded Data Shape: (328135, 169)
Updated Headers Length: 169


The last step of this "data modification" step was to get a subsample of the modified training set, and to visually inspect if there was any problem/detail we forgot:

In [22]:
def save_csv_with_headers(data, headers, file_name="data_with_headers.csv"):
    """
    Saves a dataset as a CSV file with headers as the first row.

    Parameters:
    data (np.ndarray): The dataset to save, of shape (num_samples, num_features).
    headers (list of str): List of headers for the columns.
    file_name (str): The name of the file to save. Default is "data_with_headers.csv".
    
    Returns:
    str: The path to the saved file.
    """
    # Check that the number of headers matches the number of columns in the data
    if len(headers) != data.shape[1]:
        raise ValueError("Number of headers must match the number of columns in data.")
    
    # Define header as a single comma-separated string
    header_str = ",".join(headers)
    
    # Save the file with headers
    np.savetxt(file_name, data, delimiter=",", header=header_str, comments='', fmt='%s')
    
    return file_name


In [23]:
save_csv_with_headers(encoded_data[:100,:], encoded_headers)

'data_with_headers.csv'

Some last small checks:

In [24]:
print(np.any(encoded_data == 777))
print(np.any(encoded_data == 999))

False
False


In [25]:
print(unique_values(encoded_data, 'BLOODCHO', sliced_features))
print(unique_values(encoded_data, '_PACAT1', sliced_features))
print(unique_values(encoded_data, 'MENTHLTH', sliced_features))

[0.0, 1.0]
[1.0, 2.0, 3.0, 4.0]
[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0]


## Standardize ? (only when value is non binary?)

In [26]:
def standardize(x):
    """Standardize the input data x

    Args:
        x: numpy array of shape=(num_samples, num_features)

    Returns:
        standardized data, shape=(num_samples, num_features)
    """
    # Calculate the mean and standard deviation, ignoring NaN values
    means = np.nanmean(x, axis=0)
    stds = np.nanstd(x, axis=0)

    # Handle case where standard deviation is zero to avoid division by zero
    stds = np.where(stds == 0, 1, stds)  # Replace 0 std with 1 to avoid division by zero

    # Standardize the data
    std_data = (x - means) / stds

    return std_data

In [27]:
print(encoded_data[0,:15])

[ 2.  1.  5.  0.  1.  1.  0.  1.  0. nan  1.  1.  0.  0.  0.]


In [28]:
x_standardized = standardize(encoded_data)
print(x_standardized[0,:15])

[-0.51316305 -0.37180494  0.22436556 -0.5593509   0.28000565  0.53566424
 -0.32970375 -0.4607842  -0.82125739         nan  0.36005526  0.53326516
 -0.85340967 -0.20630691 -0.39410744]


## Deal with missing data? (à dvlper?)

Input missing data by 0 (because after standardization, the mean is 0) (or remove more columns? rows?)

In [29]:
x_imputed = np.nan_to_num(x_standardized, nan=0.0)
x_imputed.shape

(328135, 169)

Check that we have no more NaN:

In [30]:
column_with_nan = find_missing_values(x_imputed)
print(column_with_nan)

{}


# TRAINING THE MODEL

In [31]:
from implementations import *

### Convert labels of y from {-1,1} to {0,1}

In [32]:
y_train[y_train == -1] = 0

# Check if all values are either 0 or 1
if np.all((y_train == 0) | (y_train == 1)):
    print("All values are either 0 or 1.")
else:
    raise ValueError("The array contains values other than 0 or 1.")

All values are either 0 or 1.


### Split the data (for now, constant split, not K-Fold)

In [33]:
def split_data(x, y, ratio, seed=1):
    """
    split the dataset based on the split ratio. If ratio is 0.8
    you will have 80% of your data set dedicated to training
    and the rest dedicated to testing. If ratio times the number of samples is not round
    you can use np.floor. Also check the documentation for np.random.permutation,
    it could be useful.

    Args:
        x: numpy array of shape (N,), N is the number of samples.
        y: numpy array of shape (N,).
        ratio: scalar in [0,1]
        seed: integer.

    Returns:
        x_tr: numpy array containing the train data.
        x_te: numpy array containing the test data.
        y_tr: numpy array containing the train labels.
        y_te: numpy array containing the test labels.
    """
    
    # set seed
    np.random.seed(seed)

    indices = np.random.permutation(x.shape[0])
    x_shuffled = x[indices]
    y_shuffled = y[indices]
    
    first_index_te = int(np.floor(x.shape[0] * ratio))
    x_tr = x_shuffled[:first_index_te]
    x_te = x_shuffled[first_index_te:]
    y_tr = y_shuffled[:first_index_te]
    y_te = y_shuffled[first_index_te:]
    return x_tr, x_te, y_tr, y_te

In [34]:
x_tr, x_val, y_tr, y_val = split_data(x_imputed, y_train, 0.8, seed=1)
print(x_tr.shape)
print(x_val.shape)
print(y_tr.shape)
print(y_val.shape)

(262508, 169)
(65627, 169)
(262508,)
(65627,)


### Train the model with penalised logistic regression

We use a penalised model to force irrelevant features i to have w_i = 0

In [35]:
lambda_ = 1
initial_w = np.zeros(x_tr.shape[1])
max_iters = 1000
gamma = 0.01

w_opt, loss_opt = reg_logistic_regression(y_tr, x_tr, lambda_, initial_w, max_iters, gamma)

Current iteration=0, loss=0.6931471805599448
Current iteration=100, loss=0.6762762734429568
Current iteration=200, loss=0.6759779130007763
Current iteration=300, loss=0.6759590643962738
loss=0.6759577361209556


In [36]:
print(w_opt)

[ 1.82607152e-02  1.18987451e-02  2.94597198e-03  5.45245753e-03
  2.43513750e-03 -9.43649518e-04  1.75528626e-03 -3.06463347e-03
  1.55095676e-02  5.10890813e-03  4.94197531e-03  7.13435776e-03
  1.39834716e-02  2.03630979e-02  2.61555104e-03  6.38810524e-03
  5.02327165e-03  1.57958263e-02  7.65864914e-03  3.51285389e-03
  1.20119990e-02  1.47689315e-02  6.13063558e-04 -1.04480116e-02
 -1.76382524e-03 -5.19089495e-03 -6.98676466e-04  1.07824416e-02
 -6.00400918e-03  3.22093424e-03  1.33808862e-03  1.13881049e-02
  1.30076855e-02  7.63503753e-03  5.48734486e-03  1.39831033e-02
  6.84080256e-03  8.28368148e-03  7.51236482e-03 -1.50812172e-03
  6.88270252e-04  1.20122030e-04 -4.80140545e-03  1.25317340e-04
  1.00482006e-04 -1.10547501e-03  1.54934001e-03  5.22076461e-04
  2.11673514e-03  7.89169985e-04  6.30283332e-04 -2.65389324e-03
 -5.45526942e-04 -2.14118482e-03  4.51119882e-03  1.13515564e-02
 -2.11368242e-03  7.03973006e-04  8.80774890e-04  1.09332310e-03
  5.57452832e-04  6.74810

### Evaluate performance on the training set

In [37]:
q = x_tr @ w_opt

limit = 0.5 #HYPERPARAM

q[q <= limit] = 0
q[q > limit] = 1

# Check if all values are either 0 or 1
if np.all((q == 0) | (q == 1)):
    print("All values are either 0 or 1.")
else:
    raise ValueError("The array contains values other than 0 or 1.")

accuracy = np.mean(y_tr == q)
print("Accuracy:", accuracy)

TP = np.sum((y_tr == 1) & (q == 1))
FP = np.sum((y_tr == 0) & (q == 1))
FN = np.sum((y_tr == 1) & (q == 0))

# Calculate Precision and Recall
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0

# Calculate F1 Score
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1_score)

All values are either 0 or 1.
Accuracy: 0.9116865009828272
Precision: 0.5111111111111111
Recall: 0.1272735072065889
F1 Score: 0.20379846824878936


### Evaluate performance on validation set

In [38]:
g = x_val @ w_opt

limit = 0.3

g[g <= limit] = 0
g[g > limit] = 1

# Check if all values are either 0 or 1
if np.all((g == 0) | (g == 1)):
    print("All values are either 0 or 1.")
else:
    raise ValueError("The array contains values other than 0 or 1.")

accuracy = np.mean(y_val == g)
print("Accuracy:", accuracy)

TP = np.sum((y_val == 1) & (g == 1))
FP = np.sum((y_val == 0) & (g == 1))
FN = np.sum((y_val == 1) & (g == 0))

# Calculate Precision and Recall
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0

# Calculate F1 Score
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1_score)

All values are either 0 or 1.
Accuracy: 0.8807198256815031
Precision: 0.3431386755542675
Recall: 0.4181529224792513
F1 Score: 0.37695001591849725


# Result Interpretation? (à dvlper?)

In [39]:
def rank_features(w, headers, N):
    """
    Ranks the features based on the coefficients from logistic regression.

    Parameters:
    w (np.ndarray): Coefficients from logistic regression.
    headers (list of str): List of feature headers.
    N (int): The number of top features to return.

    Returns:
    list of tuples: A list containing the top N features and their coefficients.
    """
    # Create a list of tuples containing feature names and their corresponding coefficients
    feature_importance = [(header, coef) for header, coef in zip(headers, w)]
    
    # Sort the list based on the absolute values of the coefficients
    feature_importance.sort(key=lambda x: abs(x[1]), reverse=True)

    # Select the top N features
    top_n_features = feature_importance[:N]

    return top_n_features

In [40]:
ranking = rank_features(w_opt, encoded_headers, 5)
print(ranking)

[('CVDSTRK3', 0.02036309794023379), ('GENHLTH', 0.018260715188943646), ('_RFHLTH', -0.018120937737925693), ('CHCCOPD1', 0.015795826283892075), ('BPHIGH4', 0.015509567577034278)]


# What could be nice to do (but sorry I'm on holydays :/)

- Apply SVM from lab 6 to our data, then compare the results with (regularized) logistic regression, to know what model should be our "ace" that we want to perfect.

- Complete the pre-processing: best way to standardize (standard? minMax could be better because often have many 0 then a few high values...), input missing values, select columns so that our best model (logreg, SVM, or other) performs even better.

- Optimise the model with hyperparameters, varying step learning rate gamma etc... Use K-fold cross-validation for that, and 
<span style="color:rgb(255, 0, 90)">/!\\ Make a plot of loss vs complexity of the model for training and val /!\\ to find the point where our model starts overfitting.</span>

- Try more advanced feature selection techniques (auto-encoder, PCA), try it on our model. If we can significantly reduce them, try implementing KNN.

- Once we have a good model, <span style="color:rgb(255, 0, 90)">/!\\ We Need a 2D plot showing some dots and squares in the feature_X-feature_Y plan, and showing the decision line /!\\.</span>

In addition, make everything nice and tidy in different files, implementations.py, helpers.py ... \
And lastly, feel free to make a lot of submissions! I feel like it's possible that F1_test < F1_val < F1_train